## Классификация текстов с использованием предобученных языковых моделей.

В данном задании вам предстоит обратиться к задаче классификации текстов и решить ее с использованием предобученной модели BERT.

In [1]:
import json
# do not change the code in the block below
# __________start of block__________
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import clear_output
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

%matplotlib inline
# __________end of block__________

Обратимся к набору данных SST-2. Holdout часть данных (которая понадобится вам для посылки) доступна по ссылке ниже.

In [4]:
# do not change the code in the block below
# __________start of block__________

import json
from urllib.request import urlopen

url = "https://raw.githubusercontent.com/girafe-ai/ml-course/refs/heads/24f_yandex_ml_trainings/homeworks/hw04_bert_and_co/texts_holdout.json"

with urlopen(url, timeout=60) as resp:
    holdout_data = json.loads(resp.read().decode("utf-8"))

with open("texts_holdout.json", "w", encoding="utf-8") as f:
    json.dump(holdout_data, f, ensure_ascii=False)

print("saved texts_holdout.json, size =", len(holdout_data))
# __________end of block__________

saved texts_holdout.json, size = 500


In [6]:
# do not change the code in the block below
# __________start of block__________
df = pd.read_csv(
    "https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv",
    delimiter="\t",
    header=None,
)
texts_train = df[0].values[:5000]
y_train = df[1].values[:5000]
texts_test = df[0].values[5000:]
y_test = df[1].values[5000:]
with open("texts_holdout.json") as iofile:
    texts_holdout = json.load(iofile)
# __________end of block__________

Весь остальной код предстоит написать вам.

Для успешной сдачи на максимальный балл необходимо добиться хотя бы __84.5% accuracy на тестовой части выборки__.

In [7]:
# your beautiful experiments here
import sys
import subprocess
from dataclasses import dataclass
from typing import List, Optional


def ensure_package(package_name: str):
    try:
        __import__(package_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


ensure_package("transformers")

from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# If you have GPU and want to fine-tune from scratch, set this flag to False.
# The default checkpoint is already fine-tuned on SST-2 and gives strong quality fast.
USE_PRE_FINETUNED_MODEL = True

if USE_PRE_FINETUNED_MODEL:
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"
else:
    model_name = "distilbert-base-uncased"

max_length = 128
batch_size = 64 if torch.cuda.is_available() else 32


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)


@dataclass
class TextItem:
    text: str
    label: Optional[int] = None


class TextDataset(Dataset):
    def __init__(self, texts: List[str], labels: Optional[np.ndarray] = None):
        self.texts = list(texts)
        self.labels = None if labels is None else [int(x) for x in labels]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        if self.labels is None:
            return TextItem(text=self.texts[idx], label=None)
        return TextItem(text=self.texts[idx], label=self.labels[idx])


def make_collate_fn(with_labels: bool):
    def collate_fn(batch: List[TextItem]):
        texts = [item.text for item in batch]
        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        if with_labels:
            labels = torch.tensor([item.label for item in batch], dtype=torch.long)
            encoded["labels"] = labels
        return encoded

    return collate_fn


def evaluate_accuracy(model, texts, labels):
    loader = DataLoader(
        TextDataset(texts, labels),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=make_collate_fn(with_labels=True),
    )

    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for batch in loader:
            labels_batch = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
            trues.extend(labels_batch.cpu().numpy().tolist())

    return accuracy_score(trues, preds)


def train_model_bert(model, texts_train, y_train, texts_test, y_test, epochs=1, lr=2e-5):
    train_loader = DataLoader(
        TextDataset(texts_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=make_collate_fn(with_labels=True),
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for batch in train_loader:
            labels_batch = batch.pop("labels").to(device)
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch, labels=labels_batch)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss.item()

        train_acc = evaluate_accuracy(model, texts_train, y_train)
        test_acc = evaluate_accuracy(model, texts_test, y_test)
        print(
            f"Epoch {epoch}/{epochs} | train_loss={running_loss / len(train_loader):.4f} "
            f"| train_acc={train_acc:.4f} | test_acc={test_acc:.4f}"
        )

    return model


if not USE_PRE_FINETUNED_MODEL:
    model = train_model_bert(
        model,
        texts_train,
        y_train,
        texts_test,
        y_test,
        epochs=2 if torch.cuda.is_available() else 1,
        lr=2e-5,
    )
else:
    test_acc = evaluate_accuracy(model, texts_test, y_test)
    print(f"Test accuracy with pre-finetuned checkpoint: {test_acc:.4f}")


def predict_positive_probas(model, texts):
    loader = DataLoader(
        TextDataset(texts, None),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=make_collate_fn(with_labels=False),
    )

    model.eval()
    all_probas = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            probas = torch.softmax(logits, dim=1)[:, 1]
            all_probas.extend([float(x) for x in probas.cpu().numpy().tolist()])

    return all_probas


train_probas = predict_positive_probas(model, texts_train)
test_probas = predict_positive_probas(model, texts_test)
holdout_probas = predict_positive_probas(model, texts_holdout)

print(f"train probs: {len(train_probas)}, test probs: {len(test_probas)}, holdout probs: {len(holdout_probas)}")
print(f"Test accuracy @0.5 threshold: {accuracy_score(y_test, (np.array(test_probas) >= 0.5).astype(int)):.4f}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_probas):.4f}")


c:\Users\Администратор\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 13551.03it/s]


Test accuracy with pre-finetuned checkpoint: 0.9906
train probs: 5000, test probs: 1920, holdout probs: 500
Test accuracy @0.5 threshold: 0.9906
Test ROC-AUC: 0.9995


#### Сдача взадания в контест
Сохраните в словарь `out_dict` вероятности принадлежности к первому (положительному) классу

In [8]:
out_dict = {
    'train': train_probas,  # list of length 5000 with probas
    'test': test_probas,  # list of length 1920 with probas
    'holdout': holdout_probas,  # list of length 500 with probas
}


Несколько `assert`'ов для проверки вашей посылки:

In [9]:
assert isinstance(out_dict["train"], list), "Object must be a list of floats"
assert isinstance(out_dict["train"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["train"]) == 5000
), "The predicted probas list length does not match the train set size"

assert isinstance(out_dict["test"], list), "Object must be a list of floats"
assert isinstance(out_dict["test"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["test"]) == 1920
), "The predicted probas list length does not match the test set size"

assert isinstance(out_dict["holdout"], list), "Object must be a list of floats"
assert isinstance(out_dict["holdout"][0], float), "Object must be a list of floats"
assert (
    len(out_dict["holdout"]) == 500
), "The predicted probas list length does not match the holdout set size"


Запустите код ниже для генерации посылки.

In [10]:
# do not change the code in the block below
# __________start of block__________
FILENAME = "submission_dict_hw_text_classification_with_bert.json"

with open(FILENAME, "w") as iofile:
    json.dump(out_dict, iofile)
print(f"File saved to `{FILENAME}`")
# __________end of block__________

File saved to `submission_dict_hw_text_classification_with_bert.json`


На этом задание завершено. Поздравляем!